In [ ]:
# =============================================================================
# FUMD-AI Training & Postprocessing Workflow -- Step 5: Map migration events (optional)
# =============================================================================
# Step:         5 of 5
# Summary:      Plot handover/migration events by type and destination cell over vehicle positions, optionally with real BS coordinates and a street map.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-training-postprocessing-workflow
# Version:      0.1.1
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# =============================================================================

# Step 5 — Map migration events (optional)

Part of the **FUMD-AI training & postprocessing workflow** (step 5 of 5,
second postprocessing step).

**Purpose.** Match each handover/migration event to the vehicle's position
at the moment it happened, then plot events grouped by type
(`case_short` -- e.g. normal handover, ping-pong) and colored by
destination cell -- generalized from `mapa_EB_K_mapAlacant.ipynb` /
`mapa_EB_K-mapAveiro.ipynb` so base-station coordinates are always a
parameter (`BS_COORDS`, a plain `{cell_id: (lon, lat)}` dict -- e.g. a
hand-curated dict of known site coordinates, the way the original
Alicante notebook worked) instead of duplicated per-city notebook
constants.

**Inputs** default to the FUMD-AI preprocessing workflow's own
`dataset_labeled_w<W>.csv` (vehicle positions) and `events_all_w<W>.csv`
(handover events) -- both directly usable as-is, so unlike the original
notebooks this needs no bespoke intermediate files
(`dataset_con_migration_v5_*.csv` / `eventos_todoss_*.csv`).

**No GPU needed.** Plain pandas/matplotlib (+ optionally `pyrosm` for the
street-map background) -- run it locally, not through `slurm/*.sbatch`.

**Outputs**, written under `OUTPUT_DIR`: `migration_events_by_type.pdf`
(and, if `OSM_PBF_PATH`/`BS_COORDS` are set, that street-map/BS-coordinate
context is overlaid on the same plot rather than a second file).


In [ ]:
import os
import sys

import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # `jupyter execute` sets the kernel's cwd to this notebook's OWN
                     # directory (see nbclient's cli.py), not the repo root -- normalize
                     # back so `sys.path.insert(0, "src")` and every relative default
                     # below resolve the same way whether this notebook is run standalone
                     # (jupyter execute notebooks/step_N...ipynb, e.g. via slurm/*.sbatch)
                     # or chained by run_pipeline.ipynb (whose own cwd is already the repo
                     # root, and which additionally sets cwd explicitly -- see
                     # fumd_training_workflow.notebook_runner.run_step).

sys.path.insert(0, "src")
from fumd_training_workflow import migration_map as mm

In [ ]:
# ---- Parameters ----
# Defaults chain onto this workflow's own bundled example slice; point
# these at a full preprocessing pipeline_run_*/ output (dataset_labeled_w<W>.csv
# + events_all_w<W>.csv) for a real run.
POSITIONS_CSV_PATH = "example-data/dataset_labeled_example.csv"
EVENTS_CSV_PATH = "example-data/events_all_example.csv"

MATCH_TOLERANCE_S = 0.05   # max time gap allowed between an event and the position sample matched to it
EXCLUDE_CASES = ["C4_no_estable"]  # diagnostic-only cases to drop before plotting
TOP_K_DESTINATIONS = 9      # destinations beyond the top K are grouped into "Otras"

# Optional: {cell_id: (lon, lat)} real coordinates to overlay -- a
# hand-curated dict of known site coordinates (see migration_map.py's
# module docstring for why building this mapping is a manual,
# domain-knowledge step, not automated). Left empty by default -- the
# plot still works from simulation coordinates alone.
BS_COORDS = {}

# Optional: a local .osm.pbf extract + pyrosm, for a real street-map
# background instead of a blank coordinate plane.
OSM_PBF_PATH = None

OUTPUT_DIR = "migration_maps"
PLOT_TITLE = "Event locations by type: destination BS"

In [ ]:
# ---- Environment variable overrides (optional) ----
# `jupyter execute` has no -p/parameter-injection flag (unlike papermill), so
# each parameter above can also be set via an environment variable of the same
# name instead of hand-editing this cell, e.g.:
#   DATASET_PATH=/abs/path/dataset_labeled_w3.csv OUTPUT_DIR=/abs/path/pipeline_run \
#   jupyter execute notebooks/step_5_map_migration_events.ipynb
# See fumd_training_workflow.notebook_runner.env_override's docstring for exact
# value-parsing rules. Unset variables leave the parameters cell's own default above untouched.
from fumd_training_workflow.notebook_runner import env_override

POSITIONS_CSV_PATH = env_override('POSITIONS_CSV_PATH', POSITIONS_CSV_PATH)
EVENTS_CSV_PATH = env_override('EVENTS_CSV_PATH', EVENTS_CSV_PATH)
MATCH_TOLERANCE_S = env_override('MATCH_TOLERANCE_S', MATCH_TOLERANCE_S)
EXCLUDE_CASES = env_override('EXCLUDE_CASES', EXCLUDE_CASES)
TOP_K_DESTINATIONS = env_override('TOP_K_DESTINATIONS', TOP_K_DESTINATIONS)
BS_COORDS = env_override('BS_COORDS', BS_COORDS)
OSM_PBF_PATH = env_override('OSM_PBF_PATH', OSM_PBF_PATH)
OUTPUT_DIR = env_override('OUTPUT_DIR', OUTPUT_DIR)
PLOT_TITLE = env_override('PLOT_TITLE', PLOT_TITLE)

## 1. Load positions and events

In [ ]:
positions = pd.read_csv(POSITIONS_CSV_PATH)
events = pd.read_csv(EVENTS_CSV_PATH)
print(f"positions: {positions.shape}, events: {events.shape}")

## 2. Match events to positions, group by destination

In [ ]:
matched = mm.match_events_to_positions(positions, events, tolerance_s=MATCH_TOLERANCE_S)
print(f"matched {len(matched)}/{len(events)} events within {MATCH_TOLERANCE_S}s")

with_case = mm.add_case_short(matched, exclude=tuple(EXCLUDE_CASES))
grouped, order_groups = mm.group_by_top_destinations(with_case, dest_col="to_cell", top_k=TOP_K_DESTINATIONS)
print("event types:", sorted(grouped["case_short"].unique()))
print("destination groups:", order_groups)

## 3. Optional street-map background

In [ ]:
edges = None
if OSM_PBF_PATH:
    from pyrosm import OSM
    lon_min = min(lon for lon, _ in BS_COORDS.values()) if BS_COORDS else grouped["x_event"].min()
    lat_min = min(lat for _, lat in BS_COORDS.values()) if BS_COORDS else grouped["y_event"].min()
    lon_max = max(lon for lon, _ in BS_COORDS.values()) if BS_COORDS else grouped["x_event"].max()
    lat_max = max(lat for _, lat in BS_COORDS.values()) if BS_COORDS else grouped["y_event"].max()
    osm = OSM(OSM_PBF_PATH, bounding_box=[lon_min, lat_min, lon_max, lat_max])
    _nodes, edges = osm.get_network(network_type="driving", nodes=True)
    print("loaded street network from", OSM_PBF_PATH)
else:
    print("OSM_PBF_PATH not set -- plotting on simulation coordinates with no street-map background")

## 4. Plot

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = os.path.join(OUTPUT_DIR, "migration_events_by_type.pdf")

mm.plot_events_by_type(
    grouped, order_groups,
    bs_coords=BS_COORDS or None, edges=edges,
    title=PLOT_TITLE, out_path=out_path,
)
print(f"saved {out_path}")